In [46]:
import pandas as pd
import numpy as np

In [47]:
file_path = "../data/processed/online_retail_II_cleaned.csv"

df = pd.read_csv(file_path)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Dataset shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

df.head()

Dataset shape: (776844, 8)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [52]:
# Calculate the total value of each transaction line

df["TotalPrice"] = df["Quantity"] * df["Price"]

print(df[["Quantity", "Price", "TotalPrice"]].head())

print("\nNegative TotalPrice:", (df["TotalPrice"] < 0).sum())
print("Zero TotalPrice:", (df["TotalPrice"] == 0).sum())

   Quantity  Price  TotalPrice
0        12   6.95        83.4
1        12   6.75        81.0
2        12   6.75        81.0
3        48   2.10       100.8
4        24   1.25        30.0

Negative TotalPrice: 0
Zero TotalPrice: 0


In [50]:
# Validate the cleaned dataset before temporal analysis

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:", df.isna().sum().sum())
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())
print("Negative prices:", (df["Price"] < 0).sum())
print("Zero prices:", (df["Price"] == 0).sum())
print("Exact duplicates:", df.duplicated().sum())

Rows: 776844
Columns: 9
Missing values: 0
Negative quantities: 0
Zero quantities: 0
Negative prices: 0
Zero prices: 0
Exact duplicates: 0


In [53]:
# Aggregate transactions into customer-month behavioral features

customer_month = (
    df.groupby(["Customer ID", "Month"])
    .agg(
        transaction_count=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_spending=("TotalPrice", "sum"),
        average_transaction_value=("TotalPrice", "mean"),
        unique_products=("StockCode", "nunique")
    )
    .reset_index()
)

customer_month.head()

,Customer ID,Month,transaction_count,total_quantity,total_spending,average_transaction_value,unique_products
0,12346.0,2010-03,1,5,27.05,5.410000,5
1,12346.0,2010-06,1,19,142.31,7.490000,19
2,12346.0,2011-01,1,74215,77183.60,77183.600000,1
3,12347.0,2010-10,1,509,611.53,15.288250,40
4,12347.0,2010-12,1,319,711.79,22.960968,31


In [54]:
# Validate the customer-month dataset

print("Customer-month rows:", len(customer_month))
print("Unique customers:", customer_month["Customer ID"].nunique())
print("Unique months:", customer_month["Month"].nunique())

print("\nMissing values:")
print(customer_month.isna().sum())

print("\nDuplicate customer-month pairs:",
      customer_month.duplicated(
          subset=["Customer ID", "Month"]
      ).sum())

Customer-month rows: 25504
Unique customers: 5853
Unique months: 25

Missing values:
Customer ID                  0
Month                        0
transaction_count            0
total_quantity               0
total_spending               0
average_transaction_value    0
unique_products              0
dtype: int64

Duplicate customer-month pairs: 0


In [55]:
# Inspect the distribution of monthly behavioral features

behavior_features = [
    "transaction_count",
    "total_quantity",
    "total_spending",
    "average_transaction_value",
    "unique_products"
]

customer_month[behavior_features].describe()

,transaction_count,total_quantity,total_spending,average_transaction_value,unique_products
count,25504.000000,25504.000000,25504.000000,25504.000000,25504.000000
mean,1.435069,411.661073,669.743128,47.782461,28.182011
std,1.288341,1681.932659,2132.380047,1168.380057,32.984161
min,1.000000,1.000000,0.850000,0.850000,1.000000
25%,1.000000,102.000000,206.622500,12.593750,10.000000
50%,1.000000,196.000000,344.245000,18.016696,19.000000
75%,1.000000,366.000000,612.425000,28.126282,35.000000
max,44.000000,93230.000000,168469.600000,168469.600000,885.000000


In [56]:
# Calculate how many months each customer was active

customer_activity = (
    customer_month.groupby("Customer ID")["Month"]
    .nunique()
)

print(customer_activity.describe())

print("\nCustomers by number of active months:")
print(customer_activity.value_counts().sort_index())

count    5853.000000
mean        4.357424
std         4.492233
min         1.000000
25%         1.000000
50%         3.000000
75%         6.000000
max        25.000000
Name: Month, dtype: float64

Customers by number of active months:
Month
1     1759
2     1073
3      659
4      495
5      384
6      281
7      195
8      185
9      149
10     115
11      89
12      75
13      53
14      60
15      38
16      48
17      32
18      28
19      33
20      27
21      16
22       9
23      15
24      15
25      20
Name: count, dtype: int64


In [57]:
# Sort customer-month observations chronologically for each customer

customer_month_sorted = customer_month.sort_values(
    ["Customer ID", "Month"]
).copy()

print(customer_month_sorted.head(10))

   Customer ID    Month  transaction_count  total_quantity  total_spending  \
0      12346.0  2010-03                  1               5           27.05   
1      12346.0  2010-06                  1              19          142.31   
2      12346.0  2011-01                  1           74215        77183.60   
3      12347.0  2010-10                  1             509          611.53   
4      12347.0  2010-12                  1             319          711.79   
5      12347.0  2011-01                  1             315          475.39   
6      12347.0  2011-04                  1             483          636.25   
7      12347.0  2011-06                  1             196          382.52   
8      12347.0  2011-08                  1             277          584.91   
9      12347.0  2011-10                  1             676         1294.32   

   average_transaction_value  unique_products  
0                   5.410000                5  
1                   7.490000               19

In [58]:
# Identify the previous active month for each customer

customer_month_sorted["previous_month"] = (
    customer_month_sorted
    .groupby("Customer ID")["Month"]
    .shift(1)
)

print(customer_month_sorted[
    ["Customer ID", "Month", "previous_month"]
].head(10))

   Customer ID    Month previous_month
0      12346.0  2010-03            NaT
1      12346.0  2010-06        2010-03
2      12346.0  2011-01        2010-06
3      12347.0  2010-10            NaT
4      12347.0  2010-12        2010-10
5      12347.0  2011-01        2010-12
6      12347.0  2011-04        2011-01
7      12347.0  2011-06        2011-04
8      12347.0  2011-08        2011-06
9      12347.0  2011-10        2011-08


In [59]:
# Calculate the number of months since the previous active month

customer_month_sorted["months_since_previous"] = (
    customer_month_sorted["Month"]
    - customer_month_sorted["previous_month"]
).apply(
    lambda x: x.n if pd.notna(x) else pd.NA
)

print(customer_month_sorted[
    ["Customer ID", "Month", "previous_month", "months_since_previous"]
].head(10))

   Customer ID    Month previous_month months_since_previous
0      12346.0  2010-03            NaT                  <NA>
1      12346.0  2010-06        2010-03                     3
2      12346.0  2011-01        2010-06                     7
3      12347.0  2010-10            NaT                  <NA>
4      12347.0  2010-12        2010-10                     2
5      12347.0  2011-01        2010-12                     1
6      12347.0  2011-04        2011-01                     3
7      12347.0  2011-06        2011-04                     2
8      12347.0  2011-08        2011-06                     2
9      12347.0  2011-10        2011-08                     2


In [60]:
# Inspect the distribution of gaps between active months

gap_data = customer_month_sorted[
    customer_month_sorted["months_since_previous"].notna()
]["months_since_previous"]

print("Customer-month observations with a previous month:", len(gap_data))

print("\nGap distribution:")
print(gap_data.describe())

print("\nGap frequencies:")
print(gap_data.value_counts().sort_index())

Customer-month observations with a previous month: 19651

Gap distribution:
count     19651
unique       23
top           1
freq       9330
Name: months_since_previous, dtype: int64

Gap frequencies:
months_since_previous
1     9330
2     3983
3     2059
4     1237
5      824
6      526
7      426
8      266
9      181
10     187
11     197
12     178
13     105
14      41
15      22
16      15
17      26
18      14
19      12
20      10
21       5
22       5
23       2
Name: count, dtype: int64


In [61]:
# Check how many observations have consecutive active months

consecutive_months = customer_month_sorted[
    customer_month_sorted["months_since_previous"] == 1
].copy()

print("Consecutive customer-month observations:", len(consecutive_months))

print(
    "Unique customers with consecutive months:",
    consecutive_months["Customer ID"].nunique()
)

Consecutive customer-month observations: 9330
Unique customers with consecutive months: 2425


In [62]:
# Create previous-month behavioral features

for feature in behavior_features:
    customer_month_sorted[f"previous_{feature}"] = (
        customer_month_sorted
        .groupby("Customer ID")[feature]
        .shift(1)
    )

print(customer_month_sorted[
    ["Customer ID", "Month"] +
    [f"previous_{feature}" for feature in behavior_features]
].head(10))

   Customer ID    Month  previous_transaction_count  previous_total_quantity  \
0      12346.0  2010-03                         NaN                      NaN   
1      12346.0  2010-06                         1.0                      5.0   
2      12346.0  2011-01                         1.0                     19.0   
3      12347.0  2010-10                         NaN                      NaN   
4      12347.0  2010-12                         1.0                    509.0   
5      12347.0  2011-01                         1.0                    319.0   
6      12347.0  2011-04                         1.0                    315.0   
7      12347.0  2011-06                         1.0                    483.0   
8      12347.0  2011-08                         1.0                    196.0   
9      12347.0  2011-10                         1.0                    277.0   

   previous_total_spending  previous_average_transaction_value  \
0                      NaN                           

In [63]:
# Calculate absolute changes in customer behavior

change_features = behavior_features.copy()

for feature in change_features:
    customer_month_sorted[f"change_{feature}"] = (
        customer_month_sorted[feature]
        - customer_month_sorted[f"previous_{feature}"]
    )

customer_month_sorted[
    ["Customer ID", "Month"] +
    [f"change_{feature}" for feature in change_features]
].head(10)

,Customer ID,Month,change_transaction_count,change_total_quantity,change_total_spending,change_average_transaction_value,change_unique_products
0,12346.0,2010-03,NaN,NaN,NaN,NaN,NaN
1,12346.0,2010-06,0.0,14.0,115.26,2.080000,14.0
2,12346.0,2011-01,0.0,74196.0,77041.29,77176.110000,-18.0
3,12347.0,2010-10,NaN,NaN,NaN,NaN,NaN
4,12347.0,2010-12,0.0,-190.0,100.26,7.672718,-9.0
5,12347.0,2011-01,0.0,-4.0,-236.40,-6.568209,-2.0
6,12347.0,2011-04,0.0,168.0,160.86,10.117658,-5.0
7,12347.0,2011-06,0.0,-287.0,-253.73,-5.259306,-6.0
8,12347.0,2011-08,0.0,81.0,202.39,5.335707,4.0
9,12347.0,2011-10,0.0,399.0,709.41,0.951905,25.0


In [64]:
# Calculate percentage changes in customer behavior

for feature in change_features:
    previous_values = customer_month_sorted[f"previous_{feature}"]
    
    customer_month_sorted[f"pct_change_{feature}"] = np.where(
        previous_values != 0,
        (
            customer_month_sorted[feature] - previous_values
        ) / previous_values * 100,
        np.nan
    )

percentage_change_features = [
    f"pct_change_{feature}"
    for feature in change_features
]

customer_month_sorted[
    ["Customer ID", "Month"] + percentage_change_features
].head(10)

,Customer ID,Month,pct_change_transaction_count,pct_change_total_quantity,pct_change_total_spending,pct_change_average_transaction_value,pct_change_unique_products
0,12346.0,2010-03,NaN,NaN,NaN,NaN,NaN
1,12346.0,2010-06,0.0,280.000000,426.099815,3.844732e+01,280.000000
2,12346.0,2011-01,0.0,390505.263158,54136.244818,1.030389e+06,-94.736842
3,12347.0,2010-10,NaN,NaN,NaN,NaN,NaN
4,12347.0,2010-12,0.0,-37.328094,16.394944,5.018702e+01,-22.500000
5,12347.0,2011-01,0.0,-1.253918,-33.212043,-2.860598e+01,-6.451613
6,12347.0,2011-04,0.0,53.333333,33.837481,6.172029e+01,-17.241379
7,12347.0,2011-06,0.0,-59.420290,-39.878978,-1.983864e+01,-25.000000
8,12347.0,2011-08,0.0,41.326531,52.909652,2.510790e+01,22.222222
9,12347.0,2011-10,0.0,144.043321,121.285326,3.580365e+00,113.636364


In [65]:
# Inspect the distribution of percentage changes

customer_month_sorted[
    percentage_change_features
].describe().T

,count,mean,std,min,25%,50%,75%,max
pct_change_transaction_count,19651.0,12.093704,62.689142,-90.000000,0.000000,0.000000,0.000000,1.300000e+03
pct_change_total_quantity,19651.0,371.370971,29177.826768,-99.951776,-43.581706,-0.689655,68.085106,4.049650e+06
pct_change_total_spending,19651.0,390.196239,41553.134675,-99.956424,-37.863187,-1.143285,52.981336,5.809197e+06
pct_change_average_transaction_value,19651.0,676.162480,83207.495527,-99.845825,-20.000000,2.090127,30.293022,1.161849e+07
pct_change_unique_products,19651.0,51.434707,303.276621,-99.462366,-40.740741,0.000000,53.333333,1.148000e+04


In [66]:
# Identify extreme percentage changes

extreme_threshold = 500

extreme_counts = (
    customer_month_sorted[percentage_change_features]
    .abs()
    .gt(extreme_threshold)
    .sum()
)

print("Observations with absolute percentage change > 500%:")
print(extreme_counts)

print("\nTotal affected observations:")
print(
    customer_month_sorted[percentage_change_features]
    .abs()
    .gt(extreme_threshold)
    .any(axis=1)
    .sum()
)

Observations with absolute percentage change > 500%:
pct_change_transaction_count             11
pct_change_total_quantity               613
pct_change_total_spending               353
pct_change_average_transaction_value    168
pct_change_unique_products              502
dtype: int64

Total affected observations:
1028


In [67]:
# Inspect extreme percentage changes using percentiles

percentile_summary = (
    customer_month_sorted[percentage_change_features]
    .quantile([0.01, 0.99])
    .T
)

percentile_summary.columns = ["P1", "P99"]

percentile_summary

,P1,P99
pct_change_transaction_count,-68.333333,200.000000
pct_change_total_quantity,-94.352226,1302.173913
pct_change_total_spending,-89.822732,766.053656
pct_change_average_transaction_value,-82.643222,455.959350
pct_change_unique_products,-92.754167,1000.000000


In [68]:
# Inspect extreme percentage changes using percentiles

percentile_summary = (
    customer_month_sorted[percentage_change_features]
    .quantile([0.01, 0.99])
    .T
)

percentile_summary.columns = ["P1", "P99"]

percentile_summary

,P1,P99
pct_change_transaction_count,-68.333333,200.000000
pct_change_total_quantity,-94.352226,1302.173913
pct_change_total_spending,-89.822732,766.053656
pct_change_average_transaction_value,-82.643222,455.959350
pct_change_unique_products,-92.754167,1000.000000
